In [1]:
import pandas as pd
import numpy as np

np.random.seed(42)
rows = 100

data = {
    "age": np.random.randint(18, 65, rows).astype(float),
    "salary": np.random.randint(20000, 120000, rows).astype(float),
    "experience": np.random.randint(0, 40, rows).astype(float),
    "credit_score": np.random.randint(300, 850, rows).astype(float),
    "city": np.random.choice(["Delhi", "Mumbai", "Bangalore", "Kolkata", None], rows),
    "gender": np.random.choice(["Male", "Female", None], rows),
    "education": np.random.choice(
        ["High School", "Bachelor", "Master", "PhD"], rows
    ),
    "marital_status": np.random.choice(
        ["Single", "Married", "Divorced"], rows
    ),
    "has_loan": np.random.choice(["Yes", "No"], rows),
    "buy": np.random.choice([0, 1], rows)
}

df = pd.DataFrame(data)

# introduce missing values
for col in ["age", "salary", "experience"]:
    df.loc[df.sample(frac=0.1).index, col] = np.nan

# introduce outliers
df.loc[df.sample(frac=0.05).index, "salary"] *= 3

# save INSIDE Colab
df.to_csv("/content/customer_data.csv", index=False)

print("Saved at /content/customer_data.csv")
df.head()


Saved at /content/customer_data.csv


,age,salary,experience,credit_score,city,gender,education,marital_status,has_loan,buy
0,56.0,NaN,11.0,803.0,Kolkata,Male,Bachelor,Single,Yes,0
1,46.0,68190.0,38.0,691.0,None,Male,Master,Single,No,0
2,32.0,NaN,1.0,434.0,Delhi,None,Master,Single,No,1
3,60.0,107538.0,NaN,494.0,Mumbai,Male,Bachelor,Married,No,0
4,25.0,59504.0,NaN,700.0,Mumbai,None,Master,Divorced,No,1


In [13]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler, StandardScaler, OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split

In [2]:
df = pd.read_csv("/content/customer_data.csv")

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 10 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   age             90 non-null     float64
 1   salary          90 non-null     float64
 2   experience      90 non-null     float64
 3   credit_score    100 non-null    float64
 4   city            74 non-null     object 
 5   gender          66 non-null     object 
 6   education       100 non-null    object 
 7   marital_status  100 non-null    object 
 8   has_loan        100 non-null    object 
 9   buy             100 non-null    int64  
dtypes: float64(4), int64(1), object(5)
memory usage: 7.9+ KB


In [4]:
df.isnull().sum()


,0
age,10
salary,10
experience,10
credit_score,0
city,26
gender,34
education,0
marital_status,0
has_loan,0
buy,0


In [48]:
df.describe()
df.shape

(100, 10)

# COLUMN CLASSIFICATION

In [6]:
num_cols = ["age", "salary", "experience", "credit_score"]
cat_nominal_cols = ["city","gender","marital_status"]
casegory_ordinal_cols = ["education"]
binary_cols = ["has_loan"]

In [ ]:
# Numerical Pipeline
num_pipeline = Pipeline(
    steps = [
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)

# Categorical Nominal Pipeline
cat_nominal_cols_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]
)   
# Categorical Ordinal Pipeline
cat_ordinal_cols_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("ordinal", OrdinalEncoder(categories=[["High School", "Bachelor", "Master", "PhD"]]))
    ]
)
# Binary Pipeline
binary_cols_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("binary", OrdinalEncoder())
    ]
)





In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", num_pipeline, num_cols),
        ("cat_nominal", cat_nominal_cols_pipeline, cat_nominal_cols),
        ("cat_ordinal", cat_ordinal_cols_pipeline, casegory_ordinal_cols),
        ("binary", binary_cols_pipeline, binary_cols)
    ]
)


In [34]:
X = df.drop("buy", axis=1)
y = df["buy"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)


In [35]:
type(X_train_processed)
X_train_processed.shape
np.isnan(X_train_processed).sum()


np.int64(0)

In [36]:
feature_names = preprocessor.get_feature_names_out()
import pandas as pd

X_train_df = pd.DataFrame(
    X_train_processed,
    columns=feature_names,
    index=X_train.index
)



In [ ]:
X_test_df = pd.DataFrame(
    X_test_processed,
    columns=feature_names,
    index=X_test.index
)

In [ ]:
# X_train_df.isna().sum().sum()
# X_test_df.isna().sum().sum()


np.int64(0)

In [ ]:
X_all_processed = preprocessor.fit_transform(X)
X_all_df = pd.DataFrame(
    X_all_processed,
    columns=preprocessor.get_feature_names_out(),
    index=X.index
)

X_all_df.info()


(100, 15)

In [49]:
X_all_df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 15 columns):
 #   Column                                Non-Null Count  Dtype  
---  ------                                --------------  -----  
 0   num__age                              100 non-null    float64
 1   num__salary                           100 non-null    float64
 2   num__experience                       100 non-null    float64
 3   num__credit_score                     100 non-null    float64
 4   cat_nominal__city_Bangalore           100 non-null    float64
 5   cat_nominal__city_Delhi               100 non-null    float64
 6   cat_nominal__city_Kolkata             100 non-null    float64
 7   cat_nominal__city_Mumbai              100 non-null    float64
 8   cat_nominal__gender_Female            100 non-null    float64
 9   cat_nominal__gender_Male              100 non-null    float64
 10  cat_nominal__marital_status_Divorced  100 non-null    float64
 11  cat_nominal__marital